# Chapter 5: Where the Action Is — Curvature

**What you'll learn:** How to compute curvature on the layer-time grid, what it means, and the key empirical finding: curvature does NOT track reasoning difficulty.

**Key concept:** Curvature measures non-commutativity — does the order of operations matter? High curvature = genuinely nonlinear computation.

**Time:** ~20 minutes

## 5.1 Setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import layer_time_geometry as core
import ltg

model = ltg.load_model("Qwen/Qwen2.5-7B", device="auto")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

## 5.2 What Is Curvature?

**Analogy:** Walk around a small square on a flat table — you end up facing the same direction (zero curvature). Walk the same square near the North Pole — you end up rotated (positive curvature).

On the layer-time grid, curvature measures: does going **right then up** give the same result as **up then right**? If not, there's curvature — the computation is genuinely nonlinear at that point.

$$\Omega(l,t) = T^{(l)}_{\text{time}} \cdot T^{(l)}_{\text{layer}} \cdot (T^{(l+1)}_{\text{time}})^{-1} \cdot (T^{(l)}_{\text{layer}})^{-1}$$

$$\text{curvature}(l,t) = \|\Omega(l,t) - I\|_F$$

In [2]:
result = ltg.analyse("Explain why the sky is blue", model=model)

# Curvature heatmap
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

im = ax1.imshow(result.curvature_map, aspect='auto', cmap='hot', origin='lower')
ax1.set_xlabel('Token position', fontsize=12)
ax1.set_ylabel('Layer', fontsize=12)
ax1.set_title('Curvature Map ||Ω(l,t) - I||', fontsize=14)
plt.colorbar(im, ax=ax1)

# Layer profile
ax2.plot(result.curvature_by_layer, color='#EE6677', linewidth=2)
ax2.fill_between(range(len(result.curvature_by_layer)), result.curvature_by_layer, alpha=0.2, color='#EE6677')
ax2.set_xlabel('Layer', fontsize=12)
ax2.set_ylabel('Mean curvature', fontsize=12)
ax2.set_title('Curvature Profile by Layer', fontsize=14)

peak = result.curvature_by_layer.argmax()
ax2.axvline(peak, color='grey', linestyle='--', alpha=0.5)
ax2.annotate(f'Peak: layer {peak}', xy=(peak, result.curvature_by_layer[peak]),
             xytext=(peak-8, result.curvature_by_layer.max()*0.8),
             fontsize=10, arrowprops=dict(arrowstyle='->'))

plt.tight_layout()
plt.show()

final3 = result.curvature_by_layer[-3:].sum() / result.curvature_by_layer.sum()
print(f"Peak curvature at layer {peak} (of {result.n_layers - 1})")
print(f"Final 3 layers contain {final3:.1%} of total curvature")

sys:1: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.


Peak curvature at layer 15 (of 27)
Final 3 layers contain 11.1% of total curvature


/var/folders/kj/3h_snmgd70v05_3h0hwqmmkw0000gn/T/ipykernel_60571/3126819575.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5.3 The Key Negative Result: Curvature ≠ Reasoning Difficulty

Early in this research, we hypothesised that harder problems would produce higher curvature. Let's test this with prompts of increasing difficulty.

In [3]:
difficulty_prompts = {
    '1-step': 'What is 5 + 3?',
    '2-step': 'What is (5 + 3) times 2?',
    '3-step': 'If x = 5 + 3 and y = x * 2, what is y - 1?',
    '4-step': 'If a = 2, b = a + 3, c = b * 2, and d = c - a, what is d?',
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

mean_curvatures = {}
for name, text in difficulty_prompts.items():
    r = ltg.analyse(text, model=model, compute_dependency=False)
    mean_curvatures[name] = r.curvature_by_layer.mean()
    ax1.plot(r.curvature_by_layer, linewidth=2, label=f'{name}: mean={r.curvature_by_layer.mean():.4f}')

ax1.set_xlabel('Layer')
ax1.set_ylabel('Mean curvature')
ax1.set_title('Curvature Profiles by Reasoning Depth')
ax1.legend(fontsize=9)

# Bar chart of mean curvatures
names = list(mean_curvatures.keys())
values = list(mean_curvatures.values())
ax2.bar(names, values, color=['#4477AA', '#66CCEE', '#CCBB44', '#EE6677'])
ax2.set_ylabel('Mean curvature')
ax2.set_title('Mean Curvature vs Reasoning Steps')

plt.tight_layout()
plt.show()

print("\nKey finding: curvature does NOT increase with reasoning depth.")
print("This was confirmed across 3 models with 290 prompts and ANOVA testing.")
print("Curvature measures non-separable interaction, NOT 'how hard the model thinks'.")


Key finding: curvature does NOT increase with reasoning depth.
This was confirmed across 3 models with 290 prompts and ANOVA testing.
Curvature measures non-separable interaction, NOT 'how hard the model thinks'.


/var/folders/kj/3h_snmgd70v05_3h0hwqmmkw0000gn/T/ipykernel_60571/174312131.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5.4 What DOES Curvature Tell Us?

Curvature doesn't track difficulty, but it DOES:
- **Concentrate in the final layers** (universally replicated)
- **Differ somewhat by task type** (with modest effect sizes)
- **Indicate non-trivial computation** (layers with zero curvature could be replaced by linear maps)

In [4]:
task_prompts = {
    'Arithmetic': 'Calculate 156 divided by 12',
    'Logic': 'All dogs are animals. Rex is a dog. Is Rex an animal?',
    'Retrieval': 'What is the capital of Japan?',
    'Definition': 'Define the word serendipity',
    'Creative': 'Write a short poem about the ocean',
}

fig, ax = plt.subplots(figsize=(10, 6))

for name, text in task_prompts.items():
    r = ltg.analyse(text, model=model, compute_dependency=False)
    curv_norm = r.curvature_by_layer / r.curvature_by_layer.sum()
    ax.plot(curv_norm, linewidth=2, label=name)

ax.set_xlabel('Layer', fontsize=12)
ax.set_ylabel('Normalised curvature', fontsize=12)
ax.set_title('Normalised Curvature Profiles by Task Type', fontsize=14)
ax.legend()
plt.tight_layout()
plt.show()

print("All tasks show the same qualitative pattern: curvature concentrates in the final layers.")

All tasks show the same qualitative pattern: curvature concentrates in the final layers.


/var/folders/kj/3h_snmgd70v05_3h0hwqmmkw0000gn/T/ipykernel_60571/1418636592.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5.5 Key Takeaways

1. **Curvature = non-commutativity** on the layer-time grid.
2. **Curvature concentrates in the final layers** — the most robust finding.
3. **Curvature does NOT track reasoning difficulty** — falsified by controlled experiments.
4. Curvature measures **non-separable interaction**, not "how hard the model thinks".

## Exercise

Design a prompt where you expect very low curvature (hint: a trivial copy/repeat task) and one where you expect high curvature (a complex multi-step problem). Analyse both. Is the difference in curvature as large as you expected?

In [5]:
# Your turn!
# r_trivial = ltg.analyse("Repeat after me: hello hello hello", model=model)
# r_complex = ltg.analyse("Complex multi-step problem...", model=model)
# Compare curvature profiles